In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
import torchaudio
import zipfile
import ast
import os

In [2]:
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/dataset.zip"
extract_path = "/content/dataset/"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP başarıyla çıkarıldı")

test_csv = '/content/drive/MyDrive/SSL_Projesi/test_metadata.csv'
df = pd.read_csv(test_csv)

print(df.head())
print("Toplam test örneği:", len(df))

Mounted at /content/drive
ZIP başarıyla çıkarıldı
                                          audio_path  azimuth_deg  distance_m  \
0  /content/dataset/dataset/session_1775030812_5c...    79.932860    2.810450   
1  /content/dataset/dataset/session_1775040605_e2...   262.116227    4.355966   
2  /content/dataset/dataset/session_1775040766_d2...    70.257374    3.220559   
3  /content/dataset/dataset/session_1775030073_23...   234.963847    3.535452   
4  /content/dataset/dataset/session_1775041687_e1...   259.779367    2.567600   

      pos_x     pos_y  pos_z  snr_db  ambient_noise_db  rt60 room_dimension  
0  2.991272  4.900000    1.5      30                15   0.3      (5, 5, 3)  
1  4.402518  0.685205    2.5       0                 0   0.3    (10, 10, 5)  
2  3.587891  4.900000    1.5      30                 0   0.5      (5, 5, 3)  
3  0.470321  0.100000    1.5      15                30   0.7      (5, 5, 3)  
4  2.044407  0.100000    1.5       0                 0   0.7      (5, 5, 

In [3]:
radius = 0.1

mics = []

for i in range(7):
    angle = 2 * np.pi * i / 7
    x = radius * np.cos(angle)
    y = radius * np.sin(angle)
    mics.append([x, y, 0])

mics.append([0, 0, 0])

mic_locs = np.array(mics).T

In [4]:
def stft_multichannel(signals, n_fft=1024, hop_length=512):
    M, N = signals.shape
    window = np.hanning(n_fft)

    n_frames = 1 + (N - n_fft) // hop_length

    stft = np.zeros((M, n_fft//2 + 1, n_frames), dtype=np.complex64)

    for m in range(M):
        for i in range(n_frames):
            start = i * hop_length
            frame = signals[m, start:start+n_fft] * window
            stft[m, :, i] = np.fft.rfft(frame, n=n_fft)

    return stft

In [5]:
def covariance_matrix(stft):
    M, F, T = stft.shape
    R = np.zeros((M, M), dtype=np.complex64)

    for f in range(F):
        X = stft[:, f, :]  # (M, T)
        R += (X @ X.conj().T) / T

    return R / F

In [6]:
def phase_mode_transform(R, mic_locs, M=7, K=7):
    angles = np.arctan2(mic_locs[1], mic_locs[0])

    m = np.arange(M)
    k = np.arange(-K, K+1)

    # Bessel-like weighting (approx)
    r = np.mean(np.linalg.norm(mic_locs[:2], axis=0))
    W = np.diag(np.sinc(k / (np.pi * r + 1e-6)))

    V = np.exp(-1j * np.outer(k, angles)) / M

    R_pm = W @ V @ R @ V.conj().T @ W

    return R_pm

In [7]:
def signal_subspace(R, n_sources=1):
    eigvals, eigvecs = np.linalg.eigh(R)
    idx = np.argsort(eigvals)[::-1]
    return eigvecs[:, idx[:n_sources]]

In [8]:
def circular_to_linear_order(mic_locs):
    """
    Çemberi açıya göre sıralar (virtual ULA üretir)
    """
    angles = np.arctan2(mic_locs[1], mic_locs[0])
    order = np.argsort(angles)

    return order

In [9]:
def esprit(U):
    U1 = U[:-1, :]
    U2 = U[1:, :]

    Phi = np.linalg.pinv(U1) @ U2
    eigvals = np.linalg.eigvals(Phi)

    return eigvals

In [10]:
def eig_to_doa(eigvals, d=0.1, wavelength=1.0):
    mu = np.angle(eigvals)

    sin_theta = (wavelength * mu) / (2 * np.pi * d)
    sin_theta = np.clip(sin_theta, -1, 1)

    return np.degrees(np.arcsin(sin_theta))

In [11]:
def estimate_wavelength(stft, fs, c=343.0):
    # En güçlü frekans bin'ini bul
    power = np.abs(stft).mean(axis=(0, 2))
    f_idx = np.argmax(power)
    # Frekansı hesapla
    freq = (f_idx * fs) / (stft.shape[1] * 2)
    freq = max(freq, 100) # Sıfıra bölünmeyi engelle
    return c / freq

In [17]:
def uca_esprit(signals, mic_locs, sr, n_fft=1024, hop_length=512):

    stft = stft_multichannel(signals, n_fft, hop_length)

    wavelength = estimate_wavelength(stft, sr)

    R = covariance_matrix(stft)

    # Phase-mode conversion (CRITICAL)
    R_pm = phase_mode_transform(R, mic_locs)

    virtual_d = 1/15

    U = signal_subspace(R_pm, n_sources=1)

    eigvals = esprit(U)

    doa = eig_to_doa(eigvals,virtual_d, wavelength)

    return doa

In [ ]:
results = []

for i in range(len(df)):
    rel_path = df.iloc[i]["audio_path"]
    true_angle = df.iloc[i]["azimuth_deg"]

    wav_path = os.path.join(extract_path, rel_path)

    mic_locs[2, :] = ast.literal_eval(df.loc[i, "room_dimension"])[2] / 2

    waveform, sr = torchaudio.load(wav_path)
    waveform = waveform.numpy()

    if waveform.shape[0] > waveform.shape[1]:
        waveform = waveform.T

    estimated_angle = uca_esprit(
        waveform[:-1, :],
        mic_locs[:,:-1],
        sr,
        n_fft=1024,
        hop_length=512
    )[0]

    estimated_angle = (np.degrees(estimated_angle)) % 360

    results.append({
        "method": "esprit",
        "file": rel_path,
        "true_angle": true_angle,
        "estimated_angle": estimated_angle,
    })

    if i % 20 == 0:
      print(f"{i}/{len(df)} işlendi")

results_df = pd.DataFrame(results)
results_df.to_csv("esprit-results.csv", index=False)

0/1620 işlendi
20/1620 işlendi
40/1620 işlendi
60/1620 işlendi
80/1620 işlendi
100/1620 işlendi
120/1620 işlendi
140/1620 işlendi
160/1620 işlendi
180/1620 işlendi
200/1620 işlendi
220/1620 işlendi
240/1620 işlendi
260/1620 işlendi
280/1620 işlendi
300/1620 işlendi
320/1620 işlendi
340/1620 işlendi
360/1620 işlendi
380/1620 işlendi
400/1620 işlendi
420/1620 işlendi
440/1620 işlendi



KeyboardInterrupt

